In [26]:
import sys
import os
import numpy as np
import torch
import pickle
from pathlib import Path
from dotenv import load_dotenv
import time

load_dotenv()

api_key = os.getenv('OPEN_AI_KEY')
print(f"API Key loaded: {'Yes' if api_key else 'No'}")

# Korrekte Pfad-Konfiguration
notebook_dir = os.getcwd()  # Aktuelles Verzeichnis (test_Project)
backend_dir = os.path.abspath(os.path.join(notebook_dir, '..', '..'))  # Backend-Verzeichnis

print(f"�� Notebook directory: {notebook_dir}")
print(f"�� Backend directory: {backend_dir}")

# Backend-Verzeichnis zum Python-Pfad hinzufügen
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)
    print(f"✅ Added {backend_dir} to Python path")

# Teste, ob modules-Verzeichnis existiert
modules_dir = os.path.join(backend_dir, 'modules')
print(f"�� Modules directory: {modules_dir}")
print(f"📁 Modules exists: {os.path.exists(modules_dir)}")

# Import our modules
try:
    from modules.KnowledgeExtraction.knowledge_extractor import KnowledgeExtractor
    from modules.KnowledgeExtraction.rag_emergency_system import RAGEmergencySystem
    from modules.KnowledgeExtraction.embedding import create_embedding
    from modules.KnowledgeExtraction.trie_structure import Trie
    from modules.KnowledgeExtraction.subgraph_builder import SubgraphBuilder
    from modules.MedicalAnalysis.emergency_detection import detect_emergency_extended, get_emergency_response_extended, analyze_medical_urgency
    print("✅ All module imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("\n🔍 Debugging information:")
    print("Available paths in sys.path:")
    for i, path in enumerate(sys.path[:5]):  # Nur die ersten 5 anzeigen
        print(f"  {i}: {path}")
    
    # Prüfe, ob modules-Verzeichnis im Backend existiert
    if os.path.exists(modules_dir):
        print(f"\n📂 Contents of modules directory:")
        try:
            for item in os.listdir(modules_dir):
                print(f"  - {item}")
        except Exception as e:
            print(f"  Error listing directory: {e}")
    else:
        print(f"\n❌ Modules directory not found at: {modules_dir}")

API Key loaded: Yes
�� Notebook directory: /Users/lucanoack/Library/CloudStorage/OneDrive-Personal/TU_Berlin/4_Semester/Programmierpraktikum/project/RobotDoc/backend/modules/test_Project
�� Backend directory: /Users/lucanoack/Library/CloudStorage/OneDrive-Personal/TU_Berlin/4_Semester/Programmierpraktikum/project/RobotDoc/backend
�� Modules directory: /Users/lucanoack/Library/CloudStorage/OneDrive-Personal/TU_Berlin/4_Semester/Programmierpraktikum/project/RobotDoc/backend/modules
📁 Modules exists: True
✅ All module imports successful!


WICHTIG: Der OPEN AI Key kostet Geld. Kann man aber sehr einfach erstellen und kostet echt nicht viel. 

Alternativ können gerne auch nach anderen Optionen geguckt werden

In [27]:
backend_path = os.path.join(os.getcwd(), '..', '..')
sys.path.insert(0, backend_path)
data_dir = os.path.join(backend_path, 'util', 'knowledgegraph_embeddings')
kg_file = os.path.join(data_dir, 'final_kg_with_embeddings_and_index_and_name.pkl')
embeddings_file = os.path.join(data_dir, 'final_kg_embeddings_tensor.pt')

print("Checking data files...")
print(f"Knowledge Graph file: {kg_file}")
print(f"Exists: {os.path.exists(kg_file)}")
print(f"Embeddings file: {embeddings_file}")
print(f"Exists: {os.path.exists(embeddings_file)}")

if os.path.exists(kg_file) and os.path.exists(embeddings_file):
    print("✅ All required data files found!")
else:
    print("❌ Missing data files. Please check the paths.")

Checking data files...
Knowledge Graph file: /Users/lucanoack/Library/CloudStorage/OneDrive-Personal/TU_Berlin/4_Semester/Programmierpraktikum/project/RobotDoc/backend/modules/test_Project/../../util/knowledgegraph_embeddings/final_kg_with_embeddings_and_index_and_name.pkl
Exists: True
Embeddings file: /Users/lucanoack/Library/CloudStorage/OneDrive-Personal/TU_Berlin/4_Semester/Programmierpraktikum/project/RobotDoc/backend/modules/test_Project/../../util/knowledgegraph_embeddings/final_kg_embeddings_tensor.pt
Exists: True
✅ All required data files found!


In [28]:
print("Initializing system components...")

# Load knowledge graph and embeddings directly
try:
    with open(kg_file, 'rb') as f:
        knowledge_graph = pickle.load(f)
    embeddings = torch.load(embeddings_file)
    print(f"✅ Knowledge graph loaded with {len(knowledge_graph)} nodes")
    print(f"✅ Embeddings loaded with shape: {embeddings.shape}")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    knowledge_graph = {}
    embeddings = None

# Initialize Trie structure
trie = Trie()
if knowledge_graph:
    # Check what type of knowledge graph we have
    if hasattr(knowledge_graph, 'nodes') and hasattr(knowledge_graph, 'edges'):
        # It's a NetworkX graph
        trie.insert_knowledge_graph(knowledge_graph, 'name', store_node_index=True)
        print("✅ Trie structure initialized with NetworkX knowledge graph")
    else:
        # It's a dictionary, we need to handle it differently
        print("⚠️  Knowledge graph is a dictionary, skipping Trie initialization")
        print(f"Knowledge graph type: {type(knowledge_graph)}")
        if knowledge_graph:
            print(f"Sample keys: {list(knowledge_graph.keys())[:5]}")
else:
    print("⚠️  Trie initialized without knowledge graph")

print("✅ Components ready for query processing")
print("Note: SubgraphBuilder and KnowledgeExtractor will be created per query as needed")

Initializing system components...
✅ Knowledge graph loaded with 161005 nodes
✅ Embeddings loaded with shape: torch.Size([161005, 1536])
✅ Trie structure initialized with NetworkX knowledge graph
✅ Components ready for query processing
Note: SubgraphBuilder and KnowledgeExtractor will be created per query as needed


In [29]:
print("=== Emergency Detection Tests ===\n")

# Test emergency detection with the actual function from emergency.py
test_queries = [
    "I have severe chest pain",
    "What is normal blood pressure?",
    "I cannot breathe",
    "How do I treat a cold?",
    "I am having a heart attack",
    "What are the symptoms of diabetes?",
    "I am bleeding severely",
    "How often should I exercise?"
]

for query in test_queries:
    is_emergency, priority, detected_keywords = detect_emergency_extended(query)
    status = "🚨 EMERGENCY" if is_emergency else "✅ Normal"
    print(f"Query: {query}")
    print(f"Status: {status} (Priority: {priority})")
    if detected_keywords:
        print(f"Detected keywords: {detected_keywords}")
    print("-" * 50)

# Test emergency response function
print("\n=== Emergency Response Tests ===\n")
emergency_queries = [
    "I have chest pain and difficulty breathing",
    "I am bleeding severely",
    "I think I'm having a stroke"
]

for query in emergency_queries:
    is_emergency, priority, detected_keywords = detect_emergency_extended(query)
    if is_emergency:
        response = get_emergency_response_extended(priority, detected_keywords)
        print(f"Query: {query}")
        print(f"Response: {response}")
        print("-" * 50)

=== Emergency Detection Tests ===

Query: I have severe chest pain
Status: 🚨 EMERGENCY (Priority: high_priority)
Detected keywords: ['chest pain']
--------------------------------------------------
Query: What is normal blood pressure?
Status: ✅ Normal (Priority: none)
--------------------------------------------------
Query: I cannot breathe
Status: ✅ Normal (Priority: none)
--------------------------------------------------
Query: How do I treat a cold?
Status: ✅ Normal (Priority: none)
--------------------------------------------------
Query: I am having a heart attack
Status: ✅ Normal (Priority: none)
--------------------------------------------------
Query: What are the symptoms of diabetes?
Status: ✅ Normal (Priority: none)
--------------------------------------------------
Query: I am bleeding severely
Status: ✅ Normal (Priority: none)
--------------------------------------------------
Query: How often should I exercise?
Status: ✅ Normal (Priority: none)
------------------------

In [30]:
print("=== RAG System Tests ===\n")

def test_rag_query(query, top_k=3):
    """Test RAG functionality for a given query"""
    try:
        # Create embedding for query
        query_embedding = create_embedding(query)
        
        # Find similar nodes
        similarities = torch.cosine_similarity(
            query_embedding.unsqueeze(0), 
            embeddings, 
            dim=1
        )
        
        # Get top-k similar nodes
        top_indices = torch.topk(similarities, min(top_k, len(similarities))).indices
        
        # Extract relevant information from NetworkX graph
        relevant_info = []
        node_list = list(knowledge_graph.nodes())
        
        for idx in top_indices:
            node_id = node_list[idx]
            node_data = knowledge_graph.nodes[node_id]
            relevant_info.append({
                'node_id': node_id,
                'info': node_data,
                'similarity': similarities[idx].item()
            })
        
        return relevant_info
    except Exception as e:
        print(f"Error in RAG query: {e}")
        return []

def create_knowledge_extractor(query_text):
    """Create a KnowledgeExtractor instance for a specific query"""
    try:
        return KnowledgeExtractor(trie=trie, text=query_text, embedding_method=create_embedding)
    except Exception as e:
        print(f"KnowledgeExtractor creation failed: {e}")
        return None

# Test RAG with medical queries
rag_test_queries = [
    "blood pressure measurement",
    "diabetes symptoms",
    "heart attack treatment",
    "first aid"
]

for query in rag_test_queries:
    print(f"Query: {query}")
    
    # Test basic RAG
    results = test_rag_query(query)
    
    if results:
        print(f"Found {len(results)} relevant results:")
        for i, result in enumerate(results, 1):
            print(f"  {i}. Node: {result['node_id']} (Similarity: {result['similarity']:.3f})")
            print(f"     Info: {result['info']}")
    else:
        print("  No relevant results found")
    
    # Test KnowledgeExtractor (optional)
    ke = create_knowledge_extractor(query)
    if ke:
        print(f"  KnowledgeExtractor created successfully")
    else:
        print(f"  KnowledgeExtractor creation failed")
    
    print("-" * 50)

=== RAG System Tests ===

Query: blood pressure measurement
Found 3 relevant results:
  1. Node: 25744 (Similarity: 0.879)
     Info: {'raw_data': "'name': 'Abnormal systemic blood pressure', 'type': 'effect/phenotype'", 'type': 'effect/phenotype', 'index': 16817, 'name': 'Abnormal systemic blood pressure', 'source': 'HPO', 'embedding': array([-0.02780643,  0.0039951 ,  0.03066099, ..., -0.03499379,
       -0.01132902, -0.02008384], dtype=float32)}
  2. Node: 94267 (Similarity: 0.874)
     Info: {'raw_data': "'name': 'Low-to-normal blood pressure', 'type': 'effect/phenotype'", 'type': 'effect/phenotype', 'index': 24012, 'name': 'Low-to-normal blood pressure', 'source': 'HPO', 'embedding': array([-0.01307973,  0.00542476,  0.04023179, ..., -0.0098255 ,
       -0.01022757, -0.02217649], dtype=float32)}
  3. Node: RedditDisease_sudden_high_blood_pressure (Similarity: 0.872)
     Info: {'type': 'RedditDisease', 'name': 'sudden high blood pressure', 'embedding': array([-0.01459626,  0.00297

In [31]:
print("=== Combined Emergency Detection + RAG System Test ===\n")

def combined_system_query(query, emergency_threshold=0.5):
    """Process query through both emergency detection and RAG"""
    
    # Step 1: Emergency Detection
    is_emergency, priority, detected_keywords = detect_emergency_extended(query)
    
    print(f"Query: {query}")
    print(f"Emergency Detection: {is_emergency} (Priority: {priority})")
    
    if is_emergency:
        print("�� EMERGENCY DETECTED - Prioritizing emergency response!")
        print(f"Detected emergency keywords: {detected_keywords}")
        
        # Get emergency response
        emergency_response_text = get_emergency_response_extended(priority, detected_keywords)
        
        # For emergencies, we might want to provide immediate guidance
        emergency_response = {
            'type': 'emergency',
            'priority': priority,
            'immediate_action': emergency_response_text,
            'keywords': detected_keywords
        }
        
        # Still provide RAG results for additional context
        rag_results = test_rag_query(query, top_k=2)
        
        return {
            'emergency_response': emergency_response,
            'rag_results': rag_results,
            'is_emergency': True
        }
    else:
        print("✅ Normal query - Processing with RAG system")
        
        # Process with full RAG system
        rag_results = test_rag_query(query, top_k=5)
        
        return {
            'emergency_response': None,
            'rag_results': rag_results,
            'is_emergency': False
        }

# Test combined system
combined_test_queries = [
    "I have severe chest pain and cannot breathe",  # Emergency
    "What is normal blood pressure?",  # Normal
    "I am bleeding severely after an accident",  # Emergency
    "How can I lower my blood pressure?",  # Normal
    "I think I'm having a heart attack",  # Emergency
    "What are the benefits of regular exercise?"  # Normal
]

for query in combined_test_queries:
    print("=" * 60)
    result = combined_system_query(query)
    
    if result['is_emergency']:
        print("\nEmergency Response:")
        print(f"  Action: {result['emergency_response']['immediate_action']}")
        print(f"  Priority: {result['emergency_response']['priority']}")
    
    if result['rag_results']:
        print("\nRAG Results:")
        for i, rag_result in enumerate(result['rag_results'], 1):
            print(f"  {i}. {rag_result['node_id']} (Similarity: {rag_result['similarity']:.3f})")
    
    print("\n" + "=" * 60 + "\n")

=== Combined Emergency Detection + RAG System Test ===

Query: I have severe chest pain and cannot breathe
Emergency Detection: True (Priority: high_priority)
�� EMERGENCY DETECTED - Prioritizing emergency response!
Detected emergency keywords: ['chest pain']

Emergency Response:
  Action: 🚨 NOTFALL ERKANNT! Die beschriebenen Symptome (chest pain) erfordern möglicherweise sofortige medizinische Hilfe. Bitte rufen Sie sofort den Notruf (112) an oder gehen Sie zum nächsten Krankenhaus. Wenn Sie bei jemandem sind, bitten Sie um sofortige Hilfe.
  Priority: high_priority

RAG Results:
  1. umls_C0423729 (Similarity: 0.931)
  2. dataset::sharp chest pain and hurts to breath. based on these symptoms (Similarity: 0.905)


Query: What is normal blood pressure?
Emergency Detection: False (Priority: none)
✅ Normal query - Processing with RAG system

RAG Results:
  1. 94267 (Similarity: 0.910)
  2. 25744 (Similarity: 0.888)
  3. 26095 (Similarity: 0.876)
  4. 120/80 (Similarity: 0.874)
  5. 86113

In [32]:
print("=== Testing Extended Emergency Detection ===\n")

from MedicalAnalysis.emergency_detection import (
    detect_emergency_extended, 
    get_emergency_response_extended, 
    analyze_medical_urgency,
    test_emergency_detection
)

print("🔍 Running built-in tests:")
test_emergency_detection()

print("\n" + "=" * 60)
print("🔍 Testing Custom German Queries:")
print("=" * 60)

custom_german_queries = [
    "Ich habe starke Brustschmerzen und kann nicht atmen",
    "Mein Kind hat hohes Fieber über 39 Grad",
    "Ich habe plötzlich starke Kopfschmerzen",
    "Was ist ein normaler Blutdruck?",
    "Ich habe einen epileptischen Anfall",
    "Ich kann nicht atmen und habe Herzschmerzen",
    "Wie behandle ich eine Erkältung?",
    "Ich habe Schwindel und Übelkeit",
    "Ich denke ich habe einen Herzinfarkt",
    "Was sind die Symptome von Diabetes?"
]

for query in custom_german_queries:
    print(f"\nQuery: {query}")
    print("-" * 40)
    
    analysis = analyze_medical_urgency(query)
    
    print(f"Emergency: {analysis['is_emergency']}")
    print(f"Priority: {analysis['priority']}")
    print(f"Urgency Score: {analysis['urgency_score']}/10")
    print(f"Symptoms: {analysis['symptoms']}")
    
    if analysis['is_emergency']:
        print(f"Response: {analysis['response']}")
        print(f"Recommendations: {analysis['recommendations']}")
    
    print("-" * 40)


print("\n" + "=" * 60)
print("�� Comparing Extended vs Original Emergency Detection:")
print("=" * 60)

comparison_queries = [
    "Ich habe starke Brustschmerzen",
    "I have chest pain",
    "Ich kann nicht atmen",
    "I cannot breathe"
]

for query in comparison_queries:
    print(f"\nQuery: {query}")
    print("-" * 40)
    
    
    is_emergency_orig, priority_orig, symptoms_orig = detect_emergency_extended(query)
    print(f"Original: Emergency={is_emergency_orig}, Priority={priority_orig}")
    
    
    is_emergency_ext, priority_ext, symptoms_ext = detect_emergency_extended(query)
    print(f"Extended: Emergency={is_emergency_ext}, Priority={priority_ext}")
    
    if is_emergency_ext:
        response = get_emergency_response_extended(priority_ext, symptoms_ext)
        print(f"Extended Response: {response}")

=== Testing Extended Emergency Detection ===

🔍 Running built-in tests:
🔍 Testing Extended Emergency Detection:

Test 1: Ich habe starke Brustschmerzen und kann kaum atmen
----------------------------------------
Emergency: True
Priority: high_priority
Urgency Score: 9/10
Symptoms: ['brustschmerzen']
Response: 🚨 NOTFALL ERKANNT! Die beschriebenen Symptome (brustschmerzen) erfordern möglicherweise sofortige medizinische Hilfe. Bitte rufen Sie sofort den Notruf (112) an oder gehen Sie zum nächsten Krankenhaus. Wenn Sie bei jemandem sind, bitten Sie um sofortige Hilfe.
Recommendations: ['Sofort Notruf 112 anrufen', 'Nächste Notaufnahme aufsuchen', 'Bei Bewusstlosigkeit: Stabile Seitenlage', 'Bei Herzstillstand: Sofortige Reanimation']


Test 2: Was ist Diabetes?
----------------------------------------
Emergency: False
Priority: none
Urgency Score: 1/10
Symptoms: []


Test 3: Ich habe einen epileptischen Anfall
----------------------------------------
Emergency: False
Priority: none
Urgen

In [33]:
print("=== Simple Extended Emergency Test ===\n")

from MedicalAnalysis.emergency_detection import analyze_medical_urgency


test_queries = [
    "Ich habe starke Brustschmerzen",
    "Was ist Diabetes?",
    "Ich kann nicht atmen",
    "Ich habe hohes Fieber"
]

for query in test_queries:
    print(f"Query: {query}")
    result = analyze_medical_urgency(query)
    print(f"Emergency: {result['is_emergency']}")
    print(f"Urgency Score: {result['urgency_score']}/10")
    if result['is_emergency']:
        print(f"Response: {result['response']}")
    print("-" * 50)

=== Simple Extended Emergency Test ===

Query: Ich habe starke Brustschmerzen
Emergency: True
Urgency Score: 9/10
Response: 🚨 NOTFALL ERKANNT! Die beschriebenen Symptome (brustschmerzen) erfordern möglicherweise sofortige medizinische Hilfe. Bitte rufen Sie sofort den Notruf (112) an oder gehen Sie zum nächsten Krankenhaus. Wenn Sie bei jemandem sind, bitten Sie um sofortige Hilfe.
--------------------------------------------------
Query: Was ist Diabetes?
Emergency: False
Urgency Score: 1/10
--------------------------------------------------
Query: Ich kann nicht atmen
Emergency: True
Urgency Score: 9/10
Response: 🚨 NOTFALL ERKANNT! Die beschriebenen Symptome (kann nicht atmen) erfordern möglicherweise sofortige medizinische Hilfe. Bitte rufen Sie sofort den Notruf (112) an oder gehen Sie zum nächsten Krankenhaus. Wenn Sie bei jemandem sind, bitten Sie um sofortige Hilfe.
--------------------------------------------------
Query: Ich habe hohes Fieber
Emergency: True
Urgency Score: 6/1

In [ ]:
print("=== Combined Extended Emergency + RAG System ===\n")

from MedicalAnalysis.emergency_detection import analyze_medical_urgency, get_emergency_response_extended

def combined_extended_system_query(query):
    """Process query through extended emergency detection and RAG"""
    
    
    analysis = analyze_medical_urgency(query)
    
    print(f"Query: {query}")
    print(f"Emergency: {analysis['is_emergency']} (Priority: {analysis['priority']})")
    print(f"Urgency Score: {analysis['urgency_score']}/10")
    
    if analysis['is_emergency']:
        print("�� EMERGENCY DETECTED - Prioritizing emergency response!")
        print(f"Detected symptoms: {analysis['symptoms']}")
        
        
        emergency_response = {
            'type': 'emergency',
            'priority': analysis['priority'],
            'urgency_score': analysis['urgency_score'],
            'immediate_action': analysis['response'],
            'recommendations': analysis['recommendations'],
            'symptoms': analysis['symptoms']
        }
        
        
        rag_results = test_rag_query(query, top_k=3)
        
        return {
            'emergency_response': emergency_response,
            'rag_results': rag_results,
            'is_emergency': True
        }
    else:
        print("✅ Normal query - Processing with RAG system")
        
        
        rag_results = test_rag_query(query, top_k=5)
        
        return {
            'emergency_response': None,
            'rag_results': rag_results,
            'is_emergency': False
        }

extended_test_queries = [
    "Ich habe starke Brustschmerzen und kann kaum atmen", 
    "I have severe chest pain and cannot breathe", 
    "Was ist normaler Blutdruck?",  
    "What is normal blood pressure?",  
    "Ich habe einen epileptischen Anfall",  
    "I am having a seizure",  
    "Wie kann ich meinen Blutdruck senken?",  
    "How can I lower my blood pressure?" 
]

for query in extended_test_queries:
    print("=" * 70)
    result = combined_extended_system_query(query)
    
    if result['is_emergency']:
        print("\n�� EMERGENCY RESPONSE:")
        print(f"  Urgency Score: {result['emergency_response']['urgency_score']}/10")
        print(f"  Action: {result['emergency_response']['immediate_action']}")
        print(f"  Recommendations:")
        for rec in result['emergency_response']['recommendations']:
            print(f"    • {rec}")
    
    if result['rag_results']:
        print("\n📚 RAG Results (Medical Information):")
        for i, rag_result in enumerate(result['rag_results'], 1):
            print(f"  {i}. {rag_result['node_id']} (Similarity: {rag_result['similarity']:.3f})")
            # Show more details for normal queries
            if not result['is_emergency']:
                print(f"     Type: {rag_result['info'].get('type', 'unknown')}")
                print(f"     Name: {rag_result['info'].get('name', 'unknown')}")
    
    print("\n" + "=" * 70 + "\n")

=== Combined Extended Emergency + RAG System ===

Query: Ich habe starke Brustschmerzen und kann kaum atmen
Emergency: True (Priority: high_priority)
Urgency Score: 9/10
�� EMERGENCY DETECTED - Prioritizing emergency response!
Detected symptoms: ['brustschmerzen']

�� EMERGENCY RESPONSE:
  Urgency Score: 9/10
  Action: 🚨 NOTFALL ERKANNT! Die beschriebenen Symptome (brustschmerzen) erfordern möglicherweise sofortige medizinische Hilfe. Bitte rufen Sie sofort den Notruf (112) an oder gehen Sie zum nächsten Krankenhaus. Wenn Sie bei jemandem sind, bitten Sie um sofortige Hilfe.
  Recommendations:
    • Sofort Notruf 112 anrufen
    • Nächste Notaufnahme aufsuchen
    • Bei Bewusstlosigkeit: Stabile Seitenlage
    • Bei Herzstillstand: Sofortige Reanimation

📚 RAG Results (Medical Information):
  1. umls_C0423729 (Similarity: 0.848)
  2. umls_C0152149 (Similarity: 0.845)
  3. concept::hard time breathing (Similarity: 0.845)


Query: I have severe chest pain and cannot breathe
Emergency: Tr

Weitere TESTS

In [ ]:
kg_path = os.path.join(backend_dir, 'util', 'knowledgegraph_embeddings', 'final_kg_with_embeddings_and_index_and_name.pkl')
emb_path = os.path.join(backend_dir, 'util', 'knowledgegraph_embeddings', 'final_kg_embeddings_tensor.pt')

print(f"�� Knowledge Graph: {kg_path}")
print(f"📁 Embeddings: {emb_path}")
print(f"✅ KG exists: {os.path.exists(kg_path)}")
print(f"✅ Embeddings exist: {os.path.exists(emb_path)}")

if os.path.exists(kg_path) and os.path.exists(emb_path):
    print("\n🚀 Loading Knowledge Graph and Embeddings...")
    
    # Load knowledge graph
    with open(kg_path, 'rb') as f:
        kg = pickle.load(f)
    
    # Load embeddings
    embeddings = torch.load(emb_path)
    
    print(f"✅ Knowledge Graph loaded: {len(kg.nodes())} nodes, {len(kg.edges())} edges")
    print(f"✅ Embeddings loaded: {embeddings.shape}")
    
    # Show sample nodes
    print("\n📊 Sample nodes:")
    for i, (node, data) in enumerate(kg.nodes(data=True)):
        if i >= 3: break
        print(f"  - {node}: {data.get('name', 'unknown')} ({data.get('type', 'unknown')})")
else:
    print("❌ Missing data files!")
    kg = None
    embeddings = None

�� Knowledge Graph: /Users/lucanoack/Library/CloudStorage/OneDrive-Personal/TU_Berlin/4_Semester/Programmierpraktikum/project/RobotDoc/backend/util/knowledgegraph_embeddings/final_kg_with_embeddings_and_index_and_name.pkl
📁 Embeddings: /Users/lucanoack/Library/CloudStorage/OneDrive-Personal/TU_Berlin/4_Semester/Programmierpraktikum/project/RobotDoc/backend/util/knowledgegraph_embeddings/final_kg_embeddings_tensor.pt
✅ KG exists: True
✅ Embeddings exist: True

🚀 Loading Knowledge Graph and Embeddings...
✅ Knowledge Graph loaded: 161005 nodes, 1895710 edges
✅ Embeddings loaded: torch.Size([161005, 1536])

📊 Sample nodes:
  - 16687: Fosinopril (drug)
  - 33577: hypertensive disorder (disease)
  - 36035: hypertension (disease)


In [36]:
print("Initializing system components...")

try:
    with open(kg_file, 'rb') as f:
        knowledge_graph = pickle.load(f)
    embeddings = torch.load(embeddings_file)
    print(f"✅ Knowledge graph loaded with {len(knowledge_graph)} nodes")
    print(f"✅ Embeddings loaded with shape: {embeddings.shape}")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    knowledge_graph = {}
    embeddings = None

# Initialize RAG Emergency System
print("\n🚀 Initializing RAG Emergency System...")
try:
    rag_system = RAGEmergencySystem(kg_file, embeddings_file)
    print("✅ RAG Emergency System initialized!")
    
    # Test basic functionality
    test_query = "What is diabetes?"
    result = rag_system.process_query(test_query)
    
    print(f"\n🧪 Test query: {test_query}")
    print(f"Emergency: {result['emergency_analysis']['is_emergency']}")
    print(f"Subgraph nodes: {result['subgraph_info']['nodes'] if result['subgraph_info'] else 'None'}")
    
except Exception as e:
    print(f"❌ Error initializing RAG system: {e}")
    import traceback
    traceback.print_exc()
    rag_system = None

trie = Trie()
if knowledge_graph:
    # Check what type of knowledge graph we have
    if hasattr(knowledge_graph, 'nodes') and hasattr(knowledge_graph, 'edges'):
        # It's a NetworkX graph
        trie.insert_knowledge_graph(knowledge_graph, 'name', store_node_index=True)
        print("✅ Trie structure initialized with NetworkX knowledge graph")
    else:
        # It's a dictionary, we need to handle it differently
        print("⚠️  Knowledge graph is a dictionary, skipping Trie initialization")
        print(f"Knowledge graph type: {type(knowledge_graph)}")
        if knowledge_graph:
            print(f"Sample keys: {list(knowledge_graph.keys())[:5]}")
else:
    print("⚠️  Trie initialized without knowledge graph")

print("✅ Components ready for query processing")

Initializing system components...
✅ Knowledge graph loaded with 161005 nodes
✅ Embeddings loaded with shape: torch.Size([161005, 1536])

🚀 Initializing RAG Emergency System...


Device set to use mps:0
Device set to use mps:0


✅ RAG Emergency System initialized!

🧪 Test query: What is diabetes?
Emergency: False
Subgraph nodes: 21
✅ Trie structure initialized with NetworkX knowledge graph
✅ Components ready for query processing


In [ ]:
def test_llm_response(query, context=None):
    """Test LLM response with optional context"""
    try:
        # Simulate LLM response
        if context:
            response = f"Based on the medical knowledge: {query}\n\nContext: {context[:200]}..."
        else:
            response = f"General response to: {query}"
        
        return {
            'query': query,
            'response': response,
            'context_used': context is not None
        }
        
    except Exception as e:
        return {
            'query': query,
            'error': str(e),
            'response': None
        }


def run_comprehensive_test(query, show_details=True):
    """Run comprehensive test for a medical query"""
    
    print(f"\n{'='*60}")
    print(f"🧪 TESTING: {query}")
    print(f"{'='*60}")
    
    results = {}

    print("\n🚨 1. Emergency Detection:")
    try:
        emergency_result = analyze_medical_urgency(query)
        results['emergency'] = emergency_result
        
        if emergency_result.get('is_emergency'):
            print(f"   �� EMERGENCY DETECTED!")
            print(f"   Priority: {emergency_result.get('priority', 'unknown')}")
            print(f"   Symptoms: {emergency_result.get('symptoms', [])}")
            print(f"   Response: {emergency_result.get('response', 'No response')}")
        else:
            print("   ✅ No emergency detected")
            
    except Exception as e:
        print(f"   ❌ Emergency detection error: {e}")
    
   
    print("\n🔍 2. RAG System (Knowledge Graph):")
    if rag_system:
        try:
            rag_result = rag_system.process_query(query)
            results['rag'] = rag_result
            
            subgraph_info = rag_result.get('subgraph_info')
            if subgraph_info:
                print(f"   📊 Subgraph: {subgraph_info['nodes']} nodes, {subgraph_info['edges']} edges")
                print(f"   🏥 Medical Entities: {len(subgraph_info['medical_entities'])}")
                
                if show_details:
                    print("   \n   Top Medical Entities:")
                    for entity in subgraph_info['medical_entities'][:5]:
                        print(f"     • {entity['name']} ({entity['type']})")
            else:
                print("   ❌ No relevant knowledge found")
                
            
            entities = rag_result.get('entities', [])
            if entities:
                print(f"   �� Extracted Entities: {len(entities)}")
                if show_details:
                    for entity in entities[:3]:
                        print(f"     • {entity}")
            
        except Exception as e:
            print(f"   ❌ RAG system error: {e}")
    else:
        print("   ❌ RAG system not available")
    
    
    print("\n🤖 3. LLM Response:")
    try:
        
        context = None
        if 'rag' in results and results['rag'].get('subgraph_info'):
            medical_entities = results['rag']['subgraph_info']['medical_entities']
            context = ", ".join([f"{e['name']} ({e['type']})" for e in medical_entities[:5]])
        
        llm_result = test_llm_response(query, context)
        results['llm'] = llm_result
        
        if llm_result.get('response'):
            print(f"   ✅ LLM Response generated")
            if show_details:
                print(f"   Response: {llm_result['response'][:200]}...")
            if llm_result.get('context_used'):
                print(f"   📚 Context used: {context[:100]}...")
        else:
            print(f"   ❌ LLM error: {llm_result.get('error', 'Unknown error')}")
            
    except Exception as e:
        print(f"   ❌ LLM test error: {e}")
    
    
    print("\n📋 4. Summary:")
    emergency_detected = results.get('emergency', {}).get('is_emergency', False)
    knowledge_found = 'rag' in results and results['rag'].get('subgraph_info') is not None
    llm_working = 'llm' in results and results['llm'].get('response') is not None
    
    print(f"   🚨 Emergency: {'Yes' if emergency_detected else 'No'}")
    print(f"   🔍 Knowledge: {'Yes' if knowledge_found else 'No'}")
    print(f"   �� LLM: {'Yes' if llm_working else 'No'}")
    
    return results

print("✅ Test functions ready!")

✅ Test functions ready!


In [38]:

test_queries = [
    "I have chest pain and difficulty breathing",
    "What is diabetes and how is it treated?",
    "Patient has high fever and severe headache",
    "I'm taking Fosinopril, what should I know?",
    "What are the symptoms of hypertension?",
    "Patient is unconscious and not breathing",
    "How can I lower my blood pressure?",
    "What is hemangioma?"
]

print("\n🧪 Running comprehensive tests...")
all_results = {}

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*20} Test {i}/{len(test_queries)} {'='*20}")
    results = run_comprehensive_test(query, show_details=True)
    all_results[query] = results
    
    # Kurze Pause zwischen Tests
    if i < len(test_queries):
        print("\n⏳ Waiting 2 seconds...")
        time.sleep(2)

print("\n✅ All tests completed!")


🧪 Running comprehensive tests...

==================== Test 1/8 ====================

🧪 TESTING: I have chest pain and difficulty breathing

🚨 1. Emergency Detection:
   �� EMERGENCY DETECTED!
   Priority: high_priority
   Symptoms: ['chest pain', 'difficulty breathing']
   Response: 🚨 NOTFALL ERKANNT! Die beschriebenen Symptome (chest pain, difficulty breathing) erfordern möglicherweise sofortige medizinische Hilfe. Bitte rufen Sie sofort den Notruf (112) an oder gehen Sie zum nächsten Krankenhaus. Wenn Sie bei jemandem sind, bitten Sie um sofortige Hilfe.

🔍 2. RAG System (Knowledge Graph):
   📊 Subgraph: 24 nodes, 20 edges
   🏥 Medical Entities: 16
   
   Top Medical Entities:
     • chest (RedditDisease)
     • Chest (UMLS)
     • pain (symptom)
     • Pain (UMLS)
     • pain (RedditSymptom)
   �� Extracted Entities: 5
     • i
     • chest
     • pain

🤖 3. LLM Response:
   ✅ LLM Response generated
   Response: Based on the medical knowledge: I have chest pain and difficulty brea

In [ ]:

print("\n📊 TEST RESULTS SUMMARY")
print("="*50)

emergency_count = 0
knowledge_count = 0
llm_count = 0

for query, results in all_results.items():
    if results.get('emergency', {}).get('is_emergency'):
        emergency_count += 1
    if 'rag' in results and results['rag'].get('subgraph_info'):
        knowledge_count += 1
    if 'llm' in results and results['llm'].get('response'):
        llm_count += 1

print(f"🚨 Emergency Detection: {emergency_count}/{len(all_results)} queries")
print(f"�� Knowledge Graph: {knowledge_count}/{len(all_results)} queries")
print(f"�� LLM Responses: {llm_count}/{len(all_results)} queries")


print("\n🚨 EMERGENCY DETECTIONS:")
for query, results in all_results.items():
    if results.get('emergency', {}).get('is_emergency'):
        priority = results['emergency'].get('priority', 'unknown')
        print(f"  • {query} ({priority})")


print("\n�� KNOWLEDGE GRAPH SUCCESS:")
for query, results in all_results.items():
    if 'rag' in results and results['rag'].get('subgraph_info'):
        nodes = results['rag']['subgraph_info']['nodes']
        entities = len(results['rag']['subgraph_info']['medical_entities'])
        print(f"  • {query} ({nodes} nodes, {entities} entities)")


📊 TEST RESULTS SUMMARY
🚨 Emergency Detection: 3/8 queries
�� Knowledge Graph: 6/8 queries
�� LLM Responses: 8/8 queries

🚨 EMERGENCY DETECTIONS:
  • I have chest pain and difficulty breathing (high_priority)
  • Patient has high fever and severe headache (medium_priority)
  • Patient is unconscious and not breathing (high_priority)

�� KNOWLEDGE GRAPH SUCCESS:
  • I have chest pain and difficulty breathing (24 nodes, 16 entities)
  • What is diabetes and how is it treated? (21 nodes, 8 entities)
  • Patient has high fever and severe headache (24 nodes, 24 entities)
  • What are the symptoms of hypertension? (19 nodes, 19 entities)
  • How can I lower my blood pressure? (10 nodes, 10 entities)
  • What is hemangioma? (20 nodes, 20 entities)


In [ ]:

your_query = "I have diabetes and chest pain"  # Ändern Sie diese Anfrage
print(f"�� Testing your query: {your_query}")

results = run_comprehensive_test(your_query, show_details=True)

print("\n📋 DETAILED RESULTS:")

emergency = results.get('emergency', {})
if emergency.get('is_emergency'):
    print(f"🚨 EMERGENCY: {emergency.get('response', 'Emergency detected')}")


rag = results.get('rag', {})
if rag.get('subgraph_info'):
    print(f"�� KNOWLEDGE: Found {len(rag['subgraph_info']['medical_entities'])} medical entities")
    for entity in rag['subgraph_info']['medical_entities'][:3]:
        print(f"   • {entity['name']} ({entity['type']})")


llm = results.get('llm', {})
if llm.get('response'):
    print(f"🤖 LLM: {llm['response'][:300]}...")

�� Testing your query: I have diabetes and chest pain

🧪 TESTING: I have diabetes and chest pain

🚨 1. Emergency Detection:
   �� EMERGENCY DETECTED!
   Priority: high_priority
   Symptoms: ['chest pain']
   Response: 🚨 NOTFALL ERKANNT! Die beschriebenen Symptome (chest pain) erfordern möglicherweise sofortige medizinische Hilfe. Bitte rufen Sie sofort den Notruf (112) an oder gehen Sie zum nächsten Krankenhaus. Wenn Sie bei jemandem sind, bitten Sie um sofortige Hilfe.

🔍 2. RAG System (Knowledge Graph):
   📊 Subgraph: 21 nodes, 20 edges
   🏥 Medical Entities: 6
   
   Top Medical Entities:
     • pain (RedditSymptom)
     • menstruation pain (RedditDisease)
     • headaches wear (RedditDisease)
     • Crippling (RedditDisease)
     • Fibrous Dysplasia (RedditDisease)
   �� Extracted Entities: 4
     • i
     • diabetes
     • chest

🤖 3. LLM Response:
   ✅ LLM Response generated
   Response: Based on the medical knowledge: I have diabetes and chest pain

Context: pain (RedditSymptom)

LLM TEST, funktionert nicht so gut lokal

##bessere lösung finden

In [43]:
# Start LLM Server
import subprocess
import time
import os

print("🚀 Starting LLM Server...")

# Navigate to llama directory
llama_dir = os.path.join(backend_dir, '..', 'llama')
print(f"LLaMA directory: {llama_dir}")

try:
    # Start the server in background
    process = subprocess.Popen(
        ['python', 'main.py'],
        cwd=llama_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    
    print("✅ LLM Server started!")
    print("⏳ Waiting 10 seconds for server to initialize...")
    time.sleep(10)
    
    # Test connection
    import socket
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(5)
            result = s.connect_ex(('localhost', 65143))
            if result == 0:
                print("✅ LLM Server is running and accessible!")
            else:
                print("❌ LLM Server is not accessible on port 65143")
    except Exception as e:
        print(f"❌ Connection test failed: {e}")
        
except Exception as e:
    print(f"❌ Failed to start LLM server: {e}")

🚀 Starting LLM Server...
LLaMA directory: /Users/lucanoack/Library/CloudStorage/OneDrive-Personal/TU_Berlin/4_Semester/Programmierpraktikum/project/RobotDoc/backend/../llama
✅ LLM Server started!
⏳ Waiting 10 seconds for server to initialize...
❌ LLM Server is not accessible on port 65143
